In [1]:
from pathlib import Path
# === Imports ===
from pathlib import Path
import torch
import torch.backends.cudnn as cudnn
import time

import sys
import os
# Add the project root to sys.path to allow imports from other folders (e.g., 'runner', 'data_loader', etc.)
sys.path.append(os.path.abspath('..'))  # Assumes this notebook is in /tutorials/

from runner.experiment import Experiment  # Main class to manage training and testing



In [ ]:
# --- Stage manifest -> pair_<d0>/ folders WGAST's loader expects -------------
#
# Source rasters live in   ./data/secondary/raw/{MODIS_d0,MODIS_t0,Landsat8_t0,Sentinel2_t0}_Istanbul/
# at native resolution (MODIS 1km, Landsat 30m, Sentinel 10m). WGAST's PatchSet
# wants them co-registered on:
#   - a 30m base grid (Landsat slot, image_size = [400,400])
#   - a 10m grid that is exactly 3x base (MODIS + Sentinel slots, 1200x1200)
#
# Filename conventions per source folder (extracted YYYYMMDD shown in brackets):
#   MODIS_t0_Istanbul     2017_06_07.tif                              [2017_06_07 -> 20170607]
#   MODIS_d0_Istanbul     2017_06_11.tif                              [2017_06_11 -> 20170611]
#   Landsat8_t0_Istanbul  LC08_180031_20170607.tif                    [last 8 chars before .tif]
#   Sentinel2_t0_Istanbul 20170607T090019_..._T35TPF.tif              [first 8 chars]

import shutil
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import Affine
from pathlib import Path

SRC = Path("./data/raw_secondary_model")
DST = Path("./data/wgast/Tdivision/test_istanbul")
DST.mkdir(parents=True, exist_ok=True)
for d in DST.glob("pair_*"):
    shutil.rmtree(d)

manifest = pd.read_parquet("./data/secondary/manifest_Istanbul.parquet") # !!!!

# Anchor grid: first Landsat scene's CRS+origin. 30m, 400x400.
ref = next((SRC / "Landsat8_t0_Istanbul").glob("*.tif"))
with rasterio.open(ref) as r:
    CRS, T30 = r.crs, r.transform
H30, W30 = 400, 400
T10 = Affine(T30.a / 3, 0, T30.c, 0, T30.e / 3, T30.f)
H10, W10 = H30 * 3, W30 * 3

def warp(src_path, dst_path, T, H, W, bands):
    with rasterio.open(src_path) as s:
        arr = np.zeros((bands, H, W), dtype=np.float32)
        for b in range(bands):
            reproject(
                source=rasterio.band(s, b + 1), destination=arr[b],
                src_transform=s.transform, src_crs=s.crs,
                dst_transform=T, dst_crs=CRS,
                resampling=Resampling.bilinear,
            )
    with rasterio.open(dst_path, "w", driver="GTiff",
                       height=H, width=W, count=bands, dtype="float32",
                       crs=CRS, transform=T) as d:
        d.write(arr)

# Index each source folder by YYYYMMDD so manifest dates match regardless of
# the original filename convention used by GEE / each processor.
modis_t0_by_date = {p.stem.replace("_", ""): p for p in (SRC / "MODIS_t0_Istanbul").glob("*.tif")}
modis_d0_by_date = {p.stem.replace("_", ""): p for p in (SRC / "MODIS_d0_Istanbul").glob("*.tif")}
landsat_by_date  = {p.stem[-8:]:             p for p in (SRC / "Landsat8_t0_Istanbul").glob("*.tif")}
s2_by_date       = {p.name[:8]:              p for p in (SRC / "Sentinel2_t0_Istanbul").glob("*.tif")}

n_ok = n_skip = 0
for _, row in manifest.iterrows():
    d0, t0 = row["d0"], row["t0"]
    fl_t0, fl_d0 = t0.strftime("%Y%m%d"), d0.strftime("%Y%m%d")
    srcs = {
        "00_MODIS":    modis_t0_by_date.get(fl_t0),
        "00_Landsat":  landsat_by_date.get(fl_t0),
        "00_Sentinel": s2_by_date.get(fl_t0),
        "01_MODIS":    modis_d0_by_date.get(fl_d0),
    }
    if not all(p and p.exists() for p in srcs.values()):
        n_skip += 1
        continue

    out = DST / f"pair_{fl_d0}"
    out.mkdir(exist_ok=True)

    warp(srcs["00_MODIS"],    out / f"00_MODIS_{fl_t0}.tif",    T10, H10, W10, 1)
    warp(srcs["00_Sentinel"], out / f"00_Sentinel_{fl_t0}.tif", T10, H10, W10, 3)
    warp(srcs["01_MODIS"],    out / f"01_MODIS_{fl_d0}.tif",    T10, H10, W10, 1)
    warp(srcs["00_Landsat"],  out / f"00_Landsat_{fl_t0}.tif",  T30, H30, W30, 4)

    # 01_Landsat@d0 stub (unused as input; only its filename is consumed)
    with rasterio.open(out / f"01_Landsat_{fl_d0}.tif", "w", driver="GTiff",
                       height=H30, width=W30, count=1, dtype="float32",
                       crs=CRS, transform=T30) as d:
        d.write(np.zeros((1, H30, W30), dtype=np.float32))

    # all-ones masks at each tif's native shape
    for f in out.glob("*.tif"):
        for sat in ("MODIS", "Landsat", "Sentinel"):
            if f"_{sat}_" in f.stem:
                with rasterio.open(f) as s:
                    shp = (s.height, s.width)
                mname = f.stem.replace(f"_{sat}_", f"_{sat}_mask_") + ".npy"
                np.save(f.parent / mname, np.ones(shp, dtype=np.float32))
                break
    n_ok += 1

print(f"Staged {n_ok} pairs into {DST}  (skipped {n_skip} rows missing an input)")

Staged 15 pairs into data/Tdivision/test_istanbul  (skipped 0 rows missing an input)


In [3]:
import os
import glob
import re
from datetime import datetime

import numpy as np
import rasterio
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset



class WGASTDataSet(Dataset):
    def __init__(self, root_dir, dem_path, date_range=None):
        self.pairs = []

        for p in sorted(Path(root_dir).glob("pair_*")):
            d = datetime.strptime(p.name.split("_")[1], "%Y%m%d")

            if date_range is None or date_range[0] <= d <= date_range[1]:
                self.pairs.append(p)

                with rasterio.open(str(dem_path)) as s:
                    self.dem_raw = s.read().astype(np.float32)

    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):

        folder = self.pairs[idx]

        day_chunks = []
        for i in range(5): #-> 5!
            s2 = self._read_tif(folder, f"{i:02d}_Sentinel_*.tif")
            ls = self._read_tif(folder, f"{i:02d}_Landsat_*.tif")
            mask = self._read_npy(folder, f"{i:02d}_Sentinel_mask_*.npy")

            H, W = s2.shape[1], s2.shape[2]
            ls = self._resize(ls, H, W)
            mask = mask[None, ...]


            s2 = s2 * mask 
            ls = ls *mask

            day_chunks.append(np.concatenate([s2,ls,mask], axis = 0))

        spatial = np.concatenate(day_chunks, axis=0)

        wgast_prev = self._read_tif(folder, "wgast_prev_*.tif")
        dem = self._resize(self.dem_raw, H,W)

        spatial = np.concatenate([spatial, wgast_prev, dem], axis=0)
        X_spatial = torch.from_numpy(spatial).float()

        return X_spatial
    @staticmethod
    def _glob_one(folder, pattern):
        hits = list(Path(folder).glob(pattern))
        if not hits:
            raise FileNotFoundError(f"{pattern} not found in {folder}")
        return hits[0]
    @staticmethod
    def _read_tif(self, folder, pattern):
        with rasterio.open(str(self._glob_one(folder, pattern))) as s:
            return s.read().astype(np.float32)


        





In [4]:
from pathlib import Path
import shutil
import sys, os
sys.path.append(os.path.abspath(".."))
from runner.experiment import Experiment

class Options:
    lr=2e-4; batch_size=1; epochs=0
    cuda=True; ngpu=1; num_workers=0
    save_dir   = Path("data/wgast/Tdivision")
    image_size = [400, 400]
    ifAdaIN=True; ifAttention=True; ifTwoInput=False
    a=1e-2; b=1; c=1; d=1
test_dir = Path("data/wgast/Tdivision/test_istanbul")
cache_dir = Path("data/secondary/wgast_cache/istanbul")
cache_dir.mkdir(parents=True, exist_ok=True)

experiment = Experiment(Options())
experiment.test(test_dir, patch_size=[32,32], num_workers=0)

for p in test_dir.glob("01_Sentinel_*.tif"):
    d0 = p.stem.split("_")[-1]                 
    shutil.move(str(p), cache_dir / f"wgast_{d0}.tif")


Model initialization
There are 7884208 trainable parameters for generator.
There are 2765505 trainable parameters for nlayerdiscriminator.
*****************
Testing...
Start test for image :  01_Sentinel_20250922.tif
Time cost: 0.09354233300109627s
End test for image :  01_Sentinel_20250922.tif
*****************************************************
Start test for image :  01_Sentinel_20251006.tif
Time cost: 0.10254670799986343s
End test for image :  01_Sentinel_20251006.tif
*****************************************************
Start test for image :  01_Sentinel_20250923.tif
Time cost: 0.10963070800062269s
End test for image :  01_Sentinel_20250923.tif
*****************************************************
Start test for image :  01_Sentinel_20251030.tif
Time cost: 0.10018925000258605s
End test for image :  01_Sentinel_20251030.tif
*****************************************************
Start test for image :  01_Sentinel_20251104.tif
Time cost: 0.10645387499971548s
End test for image :  0